<center>
<h1 style="font-family:verdana">
 🏘 Sistema de diàleg basat en regles 🏘 <h1>








ENTREGAR.

Doc pdf de max 10 pags, (ej 7: qué caminos felices hemos logrado, qué hemos arreglado) (explicación modelo llm)

(En el ej 7 hay que ver los problemas hasta el 6, y arreglarlos con reglas.)

Apendices no cuentan como parte de las 10 páginas.

Si se hace uso de IA se debe escribir en el informe, cómo se ha usado, y nosotros debemos entender el código y qué le hemos pedido.

<p>   🎯 <b>Objectiu</b>: en aquesta pràctica aprendrem a crear un assistent senzill de compra d'habitatges. El sistema haurà d'identificar i mostrar les cases que coincideixen amb les preferències de l'usuari. A poc a poc anirem afegint-hi funcionalitats, perquè el sistema siga més complet. </p>

<p> ✨ <b>Contingut</b>: en primer lloc, començarem amb un exemple senzill en què el sistema llançarà unes preguntes i respostes predefinides i l'usuari haurà d'escollir. A poc a poc anirem afegint-hi funcionalitats, perquè el sistema sigui més complet. </p>

✏ <b>Exercicis</b>: en cada secció anireu trobant exercicis que haureu d'anar resolent.

---

<h2> Índex </h2>


1. [Fitxer JSON](#section-one)
  * [Exercici 1](#ex-one)
2. [Sistema de diàleg senzill](#section-two)
  * [Exercici 2](#ex-two)
  * [Exercici 3](#ex-three)
  * [Exercici 4](#ex-four)
3. [Millorem el sistema de diàleg](#section-three)
  * [Exercici 5](#ex-five)
  * [Exercici 6](#ex-six)
4. [Lliurable](#section-four)
---


In [1]:
import json
json_path = 'house_data.json'
with open(json_path, 'r') as f:
    data = json.load(f)
print(type(data))

<class 'dict'>


In [2]:
from nltk.tokenize.treebank import TreebankWordTokenizer
import nltk
nltk.download('omw-1.4')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import TweetTokenizer

import json
import sys

from google.colab import output

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\usuar\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\usuar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\usuar\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\usuar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\usuar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


ModuleNotFoundError: No module named 'google.colab'

<h1><a name="section-one"> 1. Fitxer JSON </a></h1>

En primer lloc, carregarem el fitxer JSON per analitzar-ne l'estructura.

In [3]:
with open(json_path) as f:
  data = json.load(f)

print('Start message: ', data['start_message'])
print('End message: ', data['end_message'])
print('Questions: ', data['questions'])
print('House: ', data['houses'])

Start message:  Welcome to the House Buying Assistant!
End message:  Thank you for using the House Buying Assistant. Goodbye!
Questions:  [{'question': 'How many bedrooms do you need?', 'type': 'numerical', 'prompt': 'Enter your choice (1 - 5 bedrooms): ', 'answer_key': 'bedrooms'}, {'question': 'How many bathrooms do you need?', 'type': 'numerical', 'prompt': 'Enter your choice (1 - 3 bathrooms): ', 'answer_key': 'bathrooms'}, {'question': 'What is your budget for the house?', 'type': 'numerical', 'prompt': 'Enter your choice (1k - 999k euros):', 'answer_key': 'price'}, {'question': 'How many square meters do you need?', 'type': 'numerical', 'prompt': 'Enter your choice (30 - 200 meters)', 'answer_key': 'square_meters'}, {'question': 'Which city or neighborhood would you prefer?', 'type': 'multichoice', 'prompt': 'Enter your choice', 'answer_key': 'location'}, {'question': 'Are you looking to buy or rent?', 'type': 'multichoice', 'prompt': 'Enter your choice (sale / rent): ', 'answer_

Podeu veure com al fitxer JSON podem trobar el missatge de benvinguda i el missatge de comiat. Les diferents preguntes predefinides que el sistema demanarà a l'usuari. I les cases que l'agència té disponibles per a oferir a l'usuari.

Si no acabeu d'entendre el format, podeu copiar el contingut del fitxer JSON [aquí](https://jsonviewer.stack.hu/) per visualitzar millor l'estructura.

---



 <h1><a name="ex-one"><center> ✏ Exercici 1 ✏</a></h1>


En aquest primer exercici us demanem que afegiu una nova casa manualment al fitxer JSON amb l'id 26. La resta de camps els podeu emplenar com vulgueu.

Si ho heu fet correctament podreu veure al següent *print* les dades que heu introduït.

In [4]:
id = 26
house = data['houses'][id-1]

print(f"House ID: {house['id']}")
print(f"Bedrooms: {house['bedrooms']}")
print(f"Bathrooms: {house['bathrooms']}")
print(f"Price: {house['price']}")
print(f"Square Meters: {house['square_meters']}")
print(f"Location: {house['location']}")

House ID: 26
Bedrooms: 3
Bathrooms: 2
Price: 240k
Square Meters: 80
Location: Barcelona


---

<h1><a name="section-two"> 2. Sistema de diàleg senzill </a></h1>


En aquesta primera part crearem un sistema senzill que mostrarà les preguntes predefinides i, a continuació l'usuari haurà d'introduir la resposta. Tractarem de manera diferent les preguntes en què la resposta siga un número i les preguntes d'opció múltiple.

Intenteu entendre el funcionament, ja que treballarem sobre aquest codi.

In [5]:
def print_question(prompt, possible_options = []):
    print(prompt)
    if not len(possible_options) == 0:
      print("Options:", ", ".join(possible_options))

def initialize_available_options(house_data, available_options):
    for house in house_data['houses']:
        for key, value in house.items():
            available_options.setdefault(key, set()).add(value)

def preprocess_answer(answer):
    answer = nltk.word_tokenize(answer)
    return answer

def get_numerical_value(tok_answer):
    for token in tok_answer:
        if token.isnumeric() or token[:-1].isnumeric():
            return token
    return ''

#Exemple de frase tokenitzada
print(preprocess_answer('Vull un pis de 35k €'))

['Vull', 'un', 'pis', 'de', '35k', '€']


In [ ]:
def process_numerical_question(question):
    print_question(question['question'])
    while True:
      answer = input(question['prompt'])
      tok_answer = preprocess_answer(answer)
      value = get_numerical_value(tok_answer)
      if not value == '':
        return value

def process_multichoice_question(question, options):
    print_question(question['question'], options)
    while True:
      answer = input(question['prompt'])
      if answer in options:
        return answer

user_preferences, available_options = {}, {}
initialize_available_options(data, available_options)

for question in data['questions']:
  answer_key = question['answer_key']
  possible_options = list(available_options.get(answer_key))

  if question['type'] == 'numerical':
    answer = process_numerical_question(question)
  else:
    answer = process_multichoice_question(question, possible_options)

  user_preferences[answer_key] = answer

How many bedrooms do you need?
How many bathrooms do you need?
What is your budget for the house?
How many square meters do you need?
Which city or neighborhood would you prefer?
Options: Barcelona, Esplugues de Llobregat, L'Hospitalet de Llobregat, Santa Coloma de Gramenet
Are you looking to buy or rent?
Options: sale, rent




---


 <h1><a name="ex-two"><center> ✏ Exercici 2 ✏ </a></h1>

A continuació millorarem un poc el sistema. Us proposem que afegiu la frase de benvinguda que tenim guardada al fitxer JSON. És a dir, abans de mostrar la primera pregunta el sistema ens donarà la benvinguda.

In [17]:
# Imprimim el missatge de benvinguda
print(data['start_message'])

for question in data['questions']:
  answer_key = question['answer_key']
  possible_options = list(available_options.get(answer_key))

  if question['type'] == 'numerical':
    answer = process_numerical_question(question)
  else:
    answer = process_multichoice_question(question, possible_options)

  user_preferences[answer_key] = answer

Welcome to the House Buying Assistant!
How many bedrooms do you need?
Enter your choice (1 - 5 bedrooms): q
Enter your choice (1 - 5 bedrooms): 2
How many bathrooms do you need?
Enter your choice (1 - 3 bathrooms): 2
What is your budget for the house?
Enter your choice (1k - 999k euros):2
How many square meters do you need?
Enter your choice (30 - 200 meters)2
Which city or neighborhood would you prefer?
Options: Santa Coloma de Gramenet, L'Hospitalet de Llobregat, Barcelona, Esplugues de Llobregat
Enter your choiceBarcelona


---


 <h1><a name="ex-three"><center> ✏ Exercici 3 ✏ </a></h1>

En aquest exercici us demanem que permeteu a l'usuari abandonar el programa quan ho desitge. Per exemple, si l'usuari escriu `quit`, que el sistema s'acomiade amb l'oració de comiat que trobareu al JSON i finalitze el programa.


🙃 `sys.exit()` genera una excepció SystemExit.

In [30]:
EXIT_COMMANDS = {
    "q",
    "quit",
    "exit",
    "e",
    ":q",
    "salir",
    "s",
    "fin",
    "end",
    "stop",
    "cancel",
    "sleep"
}

def process_numerical_question(question):
    print_question(question['question'])
    while True:
      answer = input(question['prompt'])
      if answer.lower() in EXIT_COMMANDS:
        return -1
      tok_answer = preprocess_answer(answer)
      value = get_numerical_value(tok_answer)
      if not value == '':
        return value

def process_multichoice_question(question, options):
    print_question(question['question'], options)
    while True:
      answer = input(question['prompt'])
      if answer.lower() in EXIT_COMMANDS:
        return -1
      if answer in options:
        return answer

In [19]:


# Imprimim el missatge de benvinguda
print(data['start_message'])

for question in data['questions']:
  answer_key = question['answer_key']
  possible_options = list(available_options.get(answer_key))

  if question['type'] == 'numerical':
    answer = process_numerical_question(question)
  else:
    answer = process_multichoice_question(question, possible_options)

  if answer == -1:
    print(data['end_message'])
    break
  user_preferences[answer_key] = answer

Welcome to the House Buying Assistant!
How many bedrooms do you need?
Enter your choice (1 - 5 bedrooms): q
Thank you for using the House Buying Assistant. Goodbye!


Funciona també si escriviu `Quit`?

---

 <h1><a name="ex-four"><center> ✏ Exercici 4 ✏ </a></h1>

Fins ara tenim a `user_preferences` les opcions que l'usuari ha escollit.







In [20]:
user_preferences

{'bedrooms': '2',
 'bathrooms': '2',
 'price': '2',
 'square_meters': '2',
 'location': 'Barcelona'}

Per tant, ara haurem de revisar si l'agència disposa d'alguna casa disponible amb aquestes característiques. Per això us proposem completar la funció `find_suitable_houses`. Aquesta funció tornarà una llista amb tants diccionaris com cases disponibles. Per exemple si segons les preferències de l'usuari només està disponible la casa amb id 3, `suitable_house` serà:


```
[{'id': 3, 'bedrooms': '2', 'bathrooms': '1', 'price': '160k', 'square_meters': '80', 'location': 'Santa Coloma de Gramenet'}]
```

In [21]:
def find_suitable_houses(data, user_preferences):
    suitable_houses = []
    for house in data['houses']:
        suitable = True
        for key, value in user_preferences.items():
            if house[key] != value:
                suitable = False
                break
        if suitable:
            suitable_houses.append(house)

    return suitable_houses

---



Una vegada tingueu la funció `find_suitable_houses` programada, podreu executar el següent *script* que us mostrarà, si n'hi ha, la casa o cases disponibles amb les característiques que heu triat.

In [22]:
def print_suitable_houses(suitable_houses):
  if suitable_houses:
    print("\nBased on your preferences, the most suitable houses are:")
    for house in suitable_houses:
      print(f"House ID: {house['id']}")
      print(f"Type: {house['type']}")
      print("Bedrooms:", house['bedrooms'])
      print("Bathrooms:", house['bathrooms'])
      print("Price:", house['price'], "euros")
      print("Square Meters:", house['square_meters'], "m^2")
      print("Location:", house['location'])
      print()
  else:
    print("\nSorry, no suitable houses match your preferences. \n")


suitable_houses = find_suitable_houses(data, user_preferences)
print_suitable_houses(suitable_houses)


Sorry, no suitable houses match your preferences. 



<h1><a name="section-three"> 3. Millorem el sistema de diàleg </a></h1>


---


 <h1><a name="ex-five"><center> ✏ Exercici 5 ✏ </a></h1>

Tal com haureu comprovat sempre esteu obligats a triar una opció. En aquest últim exercici proposem que afegiu l'opció de no triar-ne cap. És a dir, si ens és igual el nombre d'habitacions, podrem posar per exemple `any` i el sistema ens farà la següent pregunta.

In [23]:
def process_numerical_question(question):
    print_question(question['question'])
    while True:
      answer = input(question['prompt'])
      if answer.lower() in EXIT_COMMANDS:
        return -1
      if answer.lower() == 'any':
        return 'any'
      tok_answer = preprocess_answer(answer)
      value = get_numerical_value(tok_answer)
      if not value == '':
        return value

def process_multichoice_question(question, options):
    print_question(question['question'], options)
    while True:
      answer = input(question['prompt'])
      if answer.lower() in EXIT_COMMANDS:
        return -1
      if answer.lower() == 'any':
        return 'any'
      if answer in options:
        return answer

def find_suitable_houses(data, user_preferences):
    suitable_houses = []
    for house in data['houses']:
        suitable = True
        for key, value in user_preferences.items():
            if value == 'any':
              continue
            if house[key] != value:
                suitable = False
                break
        if suitable:
            suitable_houses.append(house)

    return suitable_houses

In [24]:

# Imprimim el missatge de benvinguda
print(data['start_message'])

for question in data['questions']:
  answer_key = question['answer_key']
  possible_options = list(available_options.get(answer_key))

  if question['type'] == 'numerical':
    answer = process_numerical_question(question)
  else:
    answer = process_multichoice_question(question, possible_options)

  if answer == -1:
    print(data['end_message'])
    break
  user_preferences[answer_key] = answer

Welcome to the House Buying Assistant!
How many bedrooms do you need?
Enter your choice (1 - 5 bedrooms): q
Thank you for using the House Buying Assistant. Goodbye!


---



 <h1><a name="ex-six"><center> ✏ Exercici 6 ✏ </a></h1>

Amplieu el vostre xatbot perquè tinga en compte la següent informació sobre:

1. **Tipus d'habitatge:** Pregunteu a l'usuari si vol una casa per a compra (tipus *"sale"* a les dades) o per llogar (tipus *"rent"*). Al final de l'execució, el sistema haurá de mostrar habitatges amb el tipus seleccionat.

2. **Ingressos:** S'aconsella no dedicar més del 35% dels ingressos a l'habitatge. En cas d'escollir lloguer, preguntar a l'usuari els ingressos mensuals de la seua llar, i només mostrar aquells que estiguen per sota del 35%.

3. **Planta:** Preguntar als usuaris a quina planta volen viure. Al final de l'execució, només s'oferiràn habitatges que estiguen en aquella planta o superior.

4. **Terrassa:** Preguntar a l'usuari si vol un habitatge amb terrassa o no. En cas de voler terrassa, només s'oferiran habitatges amb terrassa. En cas contrari, oferir habitatges amb terrassa o sense.

5. **Ascensor:** Preguntar a l'usuari si vol un habitatge amb ascensor o no. En cas de voler ascensor, només s'oferiran habitatges amb ascensor. En cas contrari, oferir habitatges amb ascensor o sense.

6. **Us comercial:** Preguntar a l'usuari si vol fer servir l'habitatge com a negoci. En cas afirmatiu, només oferir habitatges amb la propietat *"commercial use"*

---





In [31]:
def to_number(value):
    """Convierte '2000' o '2k' en número."""
    value = str(value).lower()
    if value.endswith('k'):
        return float(value[:-1]) * 1000
    return float(value)

def find_suitable_houses(data, user_preferences):
    suitable_houses = []
    for house in data['houses']:
        suitable = True
        for key, value in user_preferences.items():
            if value == 'any':
                continue

            if key == 'income':
                if user_preferences.get('type') in ['rent', 'any'] and house['type'] == 'rent' and int(house['price']) > to_number(value) * 0.35:
                    suitable = False
                    break
                continue

            if key == 'floor':
                if int(house['floor']) < int(value):
                    suitable = False
                    break
                continue


            if key in ['terrace', 'elevator', 'commercial_use']:
                if value == 'Yes' and house[key] != value:
                    suitable = False
                    break
                continue

            if house[key] != value:
                suitable = False
                break

        if suitable:
            suitable_houses.append(house)

    return suitable_houses


def print_suitable_houses(suitable_houses):
  if suitable_houses:
    print("\nBased on your preferences, the most suitable houses are:")
    for house in suitable_houses:
      print(f"House ID: {house['id']}")
      print(f"Type: {house['type']}")
      print("Bedrooms:", house['bedrooms'])
      print("Bathrooms:", house['bathrooms'])
      print("Price:", house['price'], "euros")
      print("Square Meters:", house['square_meters'], "m^2")
      print("Floor:", house['floor'])
      print("Elevator:", house['elevator'])
      print("Terrace:", house['terrace'])
      print("Commercial Use:", house['commercial_use'])
      print("Location:", house['location'])
  else:
    print("\nSorry, no suitable houses match your preferences. \n")


In [32]:
# Imprimim el missatge de benvinguda
print(data['start_message'])

for question in data['questions']:
  answer_key = question['answer_key']

  if answer_key == 'income' and user_preferences.get('type') not in ['rent', 'any']:
        user_preferences['income'] = '0' #  0 com a valor per defecte
        continue

  possible_options = list(available_options.get(answer_key, []))

  if question['type'] == 'numerical':
    answer = process_numerical_question(question)
  else:
    answer = process_multichoice_question(question, possible_options)

  if answer == -1:
    print(data['end_message'])
    break
  user_preferences[answer_key] = answer

output.clear()


suitable_houses = find_suitable_houses(data, user_preferences)
print_suitable_houses(suitable_houses)


Based on your preferences, the most suitable houses are:
House ID: 26
Type: sale
Bedrooms: 3
Bathrooms: 2
Price: 240k euros
Square Meters: 80 m^2
Floor: 1
Elevator: No
Terrace: No
Commercial Use: No
Location: Barcelona


 <h1><a name="ex-seven"><center> ✏ Exercici 7 ✏ </a></h1>

Ara ja teniu un sistema funcional. Aplicant el mètode vist a classe (Exemples, Camins, Prototipat i Proves) milloreu aquest sistema per donar una millor experiència als usuaris.

---

In [ ]:
# El codi de l'Ex. 7 viu al paquet `recomanador/` (cal tenir-lo al mateix directori que house_data_v2.json).
# house_data.json queda intacte per als Ex. 1-6.

from recomanador import run
state, suitable_houses = run()